In [2]:
import mysql.connector


conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="root",
    database="palworld_database"  
)

cursor = conn.cursor()


In [3]:
cursor.execute("SHOW TABLES;")
tables = cursor.fetchall()

print("📦 Tables dans la base palworld_database :")
for t in tables:
    print("🔹", t[0])


📦 Tables dans la base palworld_database :
🔹 combat_attribute
🔹 hidden_attribute
🔹 job_skill
🔹 ordinary_boss_attribute
🔹 refresh_area
🔹 tower_boss_attribute


In [21]:
cursor.execute("DESCRIBE job_skill;")
structure = cursor.fetchall()

print("🧱 Structure de job_skill :")
for row in structure:
    print(row)


🧱 Structure de job_skill :
('id', 'int', 'NO', 'PRI', None, '')
('english_name', 'varchar(100)', 'YES', '', None, '')
('chinese_name', 'varchar(100)', 'YES', '', None, '')
('volume_size', 'text', 'YES', '', None, '')
('food', 'int', 'YES', '', None, '')
('farm_efficiency', 'float', 'YES', '', None, '')
('mining_efficiency', 'float', 'YES', '', None, '')
('logging_efficiency', 'float', 'YES', '', None, '')
('crafting_efficiency', 'float', 'YES', '', None, '')
('transport_speed', 'float', 'YES', '', None, '')
('gathering_speed', 'float', 'YES', '', None, '')
('night_shift', 'tinyint(1)', 'YES', '', None, '')
('ranch_items', 'varchar(255)', 'YES', '', None, '')
('largest_ranch_rate', 'float', 'YES', '', None, '')
('work_skill_1', 'varchar(100)', 'YES', '', None, '')
('work_skill_2', 'varchar(100)', 'YES', '', None, '')
('work_skill_3', 'varchar(100)', 'YES', '', None, '')
('special_ability', 'varchar(100)', 'YES', '', None, '')
('mount_type', 'varchar(100)', 'YES', '', None, '')
('mount_s

In [14]:
cursor.execute("ALTER TABLE hidden_attribute CHANGE size volume_size TEXT;")
conn.commit()
print("✅ Colonne 'size' renommée en 'volume_size' et convertie en TEXT.")


✅ Colonne 'size' renommée en 'volume_size' et convertie en TEXT.


In [15]:
# Liste des requêtes


alter_queries = [
    "ALTER TABLE combat_attribute MODIFY volume_size TEXT;",
    "ALTER TABLE job_skill MODIFY volume_size TEXT;",
    "ALTER TABLE hidden_attribute MODIFY volume_size TEXT;"
]

# Exécution de chaque requête
for query in alter_queries:
    try:
        cursor.execute(query)
        print(f"✅ OK : {query}")
    except Exception as e:
        print(f"❌ Erreur avec : {query}\n{e}")

conn.commit()


✅ OK : ALTER TABLE combat_attribute MODIFY volume_size TEXT;
✅ OK : ALTER TABLE job_skill MODIFY volume_size TEXT;
✅ OK : ALTER TABLE hidden_attribute MODIFY volume_size TEXT;


In [17]:
def drop_empty_columns(table_name):
    cursor.execute(f"DESCRIBE hidden_attribute")
    columns = [row[0] for row in cursor.fetchall()]

    dropped = []

    for col in columns:
        # Test si la colonne est entièrement NULL
        cursor.execute(f"SELECT COUNT(*) FROM  hidden_attribute WHERE `{col}` IS NOT NULL")
        count = cursor.fetchone()[0]

        if count == 0:
            cursor.execute(f"ALTER TABLE hidden_attribute DROP COLUMN `{col}`")
            dropped.append(col)

    conn.commit()
    return dropped


In [18]:
cursor.execute("SHOW TABLES")
tables = [row[0] for row in cursor.fetchall()]

for table in tables:
    removed = drop_empty_columns(table)
    if removed:
        print(f"🧹 Colonnes supprimées dans {table} : {removed}")
    else:
        print(f"✅ Aucune colonne vide à supprimer dans {table}")


✅ Aucune colonne vide à supprimer dans combat_attribute
✅ Aucune colonne vide à supprimer dans hidden_attribute
✅ Aucune colonne vide à supprimer dans job_skill
✅ Aucune colonne vide à supprimer dans ordinary_boss_attribute
✅ Aucune colonne vide à supprimer dans refresh_area
✅ Aucune colonne vide à supprimer dans tower_boss_attribute


In [20]:
import pandas as pd

# Charger le fichier
df = pd.read_csv("tables/clean_job_skill_final.csv")

# Nettoyage : convertir les colonnes numériques
cols_float = [
    "farm_efficiency", "mining_efficiency", "logging_efficiency", 
    "crafting_efficiency", "transport_speed", "gathering_speed", 
    "largest_ranch_rate", "mount_speed", "flying_speed", "swim_speed"
]

# Convertir en float et forcer les erreurs à devenir NaN
for col in cols_float:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Nettoyage des booléens
if "night_shift" in df.columns:
    df["night_shift"] = df["night_shift"].astype(str).str.lower().map({
        "true": 1, "false": 0, "1": 1, "0": 0, "yes": 1, "no": 0
    }).fillna(0).astype(int)

# Remplacer les NaN (optionnel, selon ton cas)
df = df.fillna("")

# Sauvegarder le fichier nettoyé
df.to_csv("clean_job_skill_final_ready.csv", index=False)

print("✅ Fichier CSV nettoyé prêt pour l'import.")


✅ Fichier CSV nettoyé prêt pour l'import.


In [23]:
cursor.execute("DROP TABLE IF EXISTS job_skill;")
conn.commit()

print("✅ Table job_skill supprimée.")

✅ Table job_skill supprimée.


In [24]:
cursor.execute("DROP TABLE IF EXISTS combat_attribute;")
conn.commit()

print("✅ Table combat_attribute  supprimée.")


✅ Table combat_attribute  supprimée.


In [25]:
cursor.execute("DROP TABLE IF EXISTS refresh_area;")
conn.commit()

print("✅ Table refresh_area  supprimée.")


✅ Table refresh_area  supprimée.


In [26]:
cursor.execute("DROP TABLE IF EXISTS hidden_attribute;")
conn.commit()

print("✅ Table hidden_attribute  supprimée.")

✅ Table hidden_attribute  supprimée.


In [27]:
cursor.execute("DROP TABLE IF EXISTS hidden_attribute;")
conn.commit()

print("✅ Table hidden_attribute  supprimée.")

✅ Table hidden_attribute  supprimée.


In [30]:


# Charger le fichier
df = pd.read_csv("tables/clean_hidden_attribute_final.csv")

# Convertir tous les booléens "True"/"False" en 1/0
df = df.replace({"True": 1, "False": 0})

# Sauvegarder un nouveau fichier corrigé
df.to_csv("tables/clean_hidden_attribute_final_ready.csv", index=False)


In [33]:
# Charger le CSV (le bon, sans colonne id)
df = pd.read_csv("tables/clean_hidden_attribute_final.csv")

# Importer dans la table (append = ajouter, if_exists='replace' = écraser)
df.to_sql(name="hidden_attribute", con=engine, if_exists="replace", index=False)

print("✅ Importation terminée sans erreur.")

NameError: name 'engine' is not defined